# Faster R-CNN: train → predict → evaluate → demo
Chạy các cell theo thứ tự trên Kaggle hoặc Jupyter local. Chọn một exp ở cell cấu hình; không cần copy cell cho từng exp.

Repo phải chứa bản code mới (`train.py`, `predict.py`, `engine.py`). Nếu dùng Kaggle, push code đã kiểm tra lên GitHub hoặc upload bản repo mới trước. Notebook không sửa source code và không hỗ trợ resume; train bắt đầu một lần chạy mới. Best checkpoint hiện chọn theo validation loss.

Train/demo mặc định tắt. Chỉ bật train khi có đủ ảnh train/val; không dùng test làm dữ liệu huấn luyện chính thức.

In [ ]:
# 1. Chọn repository; dùng checkout hiện có hoặc clone nếu chưa có.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tmcng/deeplearning-project-sign-detection.git"
REPO_REF = "main"  # Nhánh/commit chứa code mới đã push.
candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / "faster_rcnn/train.py").is_file()), None)
if REPO is None:
    REPO = Path("/kaggle/working/deeplearning-project-sign-detection")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
for relative in ("faster_rcnn/predict.py", "faster_rcnn/engine.py", "streaming/inference.py"):
    if not (REPO / relative).is_file():
        raise RuntimeError(f"Repo thiếu {relative}. Upload/push code mới trước khi chạy.")
print("Repository:", REPO)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
# 2. Cài dependencies khi cần; giữ nguyên cặp torch/torchvision của môi trường.
INSTALL_DEPENDENCIES = False  # Bật một lần nếu môi trường còn thiếu thư viện.
if INSTALL_DEPENDENCIES:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "pyyaml", "tqdm", "opencv-python-headless", "pycocotools",
        "wandb", "psutil", "GPUtil",
    ], check=True)

import torch
import torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

## Cấu hình chung
Sửa đường dẫn ảnh/nhãn cho dataset của bạn. Các đường dẫn bên dưới là ví dụ Kaggle từ notebook cũ.
`WEIGHTS_OVERRIDE` nhận file state_dict .pth có sẵn; bỏ trống để dùng best checkpoint của exp.
Nếu Kaggle đã giải nén weights thành thư mục, hãy cung cấp lại file .pth hợp lệ; notebook không đóng gói ZIP hay đổi đuôi để đoán định dạng.

In [ ]:
# 3. Chọn exp, dữ liệu và các tác vụ cần chạy.
EXPERIMENT = "exp03"
EXPERIMENTS = {
    "exp01": ("exp01_baseline", "fasterrcnn_baseline"),
    "exp02": ("exp02_fpn_small_anchor", "fasterrcnn_fpn_small_anchor"),
    "exp03": ("exp03_fpn_aug", "fasterrcnn_fpn_aug"),
}
EXP_NAME, CHECKPOINT_NAME = EXPERIMENTS[EXPERIMENT]
CONFIG = REPO / "faster_rcnn/configs/experiment" / f"{EXP_NAME}.yaml"
IMAGE_DIR = Path("/kaggle/input/datasets/phhasian0710/za-traffic-2020/za_traffic_2020/traffic_train/images")
LABEL_DIR = Path("/kaggle/input/datasets/hpwj123/labeldataset")
SPLIT = "val"  # Chuyển test sau khi chốt mô hình.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4 if DEVICE == "cuda" else 1
NUM_WORKERS = 2 if DEVICE == "cuda" else 0
EPOCHS = 30
RUN_SMOKE_TRAIN = False
RUN_TRAIN = False
RUN_PREDICT = True
RUN_EVALUATE = True
RUN_DEMO = False
USE_WANDB = False
WEIGHTS_OVERRIDE = None  # Ví dụ Path("/kaggle/input/my-weights/model.pth")
VIDEO_SOURCE = None  # Ví dụ "/kaggle/input/my-video/traffic.mp4"

CHECKPOINT_DIR = REPO / "faster_rcnn/checkpoints"
WEIGHTS = Path(WEIGHTS_OVERRIDE) if WEIGHTS_OVERRIDE else CHECKPOINT_DIR / f"{CHECKPOINT_NAME}_best.pth"
PRED_PATH = REPO / "preds" / f"{EXP_NAME}_{SPLIT}.json"
REPORT_DIR = REPO / "reports/eval"
assert SPLIT in ("val", "test")
print(EXP_NAME, DEVICE, "| weights:", WEIGHTS)

In [ ]:
# 4. Kiểm tra ảnh được tham chiếu trong JSON trước tác vụ; không tạo symlink.
import json

required_splits = set()
if RUN_TRAIN or RUN_SMOKE_TRAIN:
    required_splits.update(("train", "val"))
if RUN_PREDICT or RUN_EVALUATE:
    required_splits.add(SPLIT)
for split in sorted(required_splits):
    annotation_path = LABEL_DIR / f"{split}.json"
    data = json.loads(annotation_path.read_text(encoding="utf-8"))
    if not data.get("images"):
        raise ValueError(f"Không có images: {annotation_path}")
    needs_images = RUN_PREDICT and split == SPLIT or (RUN_TRAIN or RUN_SMOKE_TRAIN) and split in ("train", "val")
    if needs_images:
        missing = [im["file_name"] for im in data["images"] if not (IMAGE_DIR / im["file_name"]).is_file()]
        if missing:
            raise FileNotFoundError(f"{split}: thiếu {len(missing)} ảnh, ví dụ {missing[:3]}")
    print(split, ":", len(data["images"]), "ảnh")

def run(*args):
    command = [sys.executable, "-X", "utf8", *map(str, args)]
    print(" ".join(command))
    subprocess.run(command, cwd=REPO, check=True)

In [ ]:
# 5. Tracking tùy chọn; không in API key hoặc truyền key qua command line.
if USE_WANDB:
    if not os.environ.get("WANDB_API_KEY"):
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("wandb")
    import wandb
    wandb.login()
else:
    print("W&B disabled")

## Train
Smoke chạy một batch và không lưu weights. Train đầy đủ dùng train/val riêng, lưu best và last.
Không có `--resume` trong CLI hiện tại. Nếu chỉ muốn dùng weights cũ để predict/demo, giữ `RUN_TRAIN=False`.

In [ ]:
# 6. Smoke train và/hoặc training đầy đủ theo lựa chọn ở trên.
train_args = [
    "-m", "faster_rcnn.train", "-c", CONFIG,
    "--img-dir", IMAGE_DIR,
    "--train-json", LABEL_DIR / "train.json",
    "--val-json", LABEL_DIR / "val.json",
    "--device", DEVICE, "--batch-size", BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--mixed-precision" if DEVICE == "cuda" else "--no-mixed-precision",
    "--checkpoint-dir", CHECKPOINT_DIR,
]
if RUN_SMOKE_TRAIN:
    run(*train_args, "--dry-run", "--no-wandb")
if RUN_TRAIN:
    run(*train_args, "--epochs", EPOCHS, "--save-best", *([] if USE_WANDB else ["--no-wandb"]))
    if WEIGHTS_OVERRIDE is not None:
        print("Lưu ý: predict vẫn dùng WEIGHTS_OVERRIDE, không dùng weights vừa train.")

In [ ]:
# 7. Inference trên toàn bộ split đã chọn, chỉ xuất COCO JSON.
if RUN_PREDICT:
    if not WEIGHTS.is_file():
        raise FileNotFoundError(f"Chưa có weights: {WEIGHTS}. Train hoặc chọn WEIGHTS_OVERRIDE.")
    run(
        "-m", "faster_rcnn.predict", "-c", CONFIG,
        "--weights", WEIGHTS, "--img-dir", IMAGE_DIR,
        "--gt", LABEL_DIR / f"{SPLIT}.json", "--split", SPLIT,
        "--device", DEVICE, "--batch-size", BATCH_SIZE,
        "--num-workers", NUM_WORKERS, "--conf", "0.001",
        "--no-dry-run", "--out-pred", PRED_PATH,
    )

In [ ]:
# 8. Chấm mAP bằng evaluator chung, độc lập với Faster R-CNN.
# Mỗi lần chạy sẽ thêm một dòng vào results.md.
if RUN_EVALUATE:
    if not PRED_PATH.is_file():
        raise FileNotFoundError(f"Chưa có predictions: {PRED_PATH}")
    run(
        "evaluation/evaluate.py",
        "--gt", LABEL_DIR / f"{SPLIT}.json",
        "--pred", PRED_PATH, "--name", f"{EXP_NAME}_{SPLIT}",
        "--out", REPORT_DIR,
    )

In [ ]:
# 9. Demo video tùy chọn: headless cho notebook, output riêng mỗi lần chạy.
from datetime import datetime
if RUN_DEMO:
    if not VIDEO_SOURCE or not WEIGHTS.is_file():
        raise ValueError("Cần VIDEO_SOURCE và file WEIGHTS hợp lệ.")
    demo_dir = REPO / "demo_outputs" / f"{EXP_NAME}_{datetime.now():%Y%m%d_%H%M%S_%f}"
    run(
        "-m", "streaming.inference", "--model", "faster-rcnn",
        "--config", CONFIG, "--weights", WEIGHTS,
        "--source", VIDEO_SOURCE, "--device", DEVICE,
        "--max-frames", 300, "--warmup", 10,
        "--conf", "0.5", "--roi-preset", "none",
        "--headless", "--save-video", "--output-dir", demo_dir,
    )
    from IPython.display import Video, display
    if (demo_dir / "demo.mp4").is_file():
        display(Video(str(demo_dir / "demo.mp4"), embed=True))

## Kết quả
- Weights: `faster_rcnn/checkpoints/<checkpoint_name>_{best,last}.pth`.
- Predictions: `preds/<exp_name>_<split>.json`.
- Báo cáo: `reports/eval/<exp_name>_<split>.json` và `results.md`.
- Demo tùy chọn: `demo_outputs/<exp_name>_<timestamp>/`.

Trên Kaggle, lưu/download các file trong /kaggle/working trước khi kết thúc session.
Chọn exp01/exp02 ở cell cấu hình rồi chạy lại từ đó; không cần sửa source code.